# Solution: Linear Regression for Mintos Investors

**Main application:** Capital invested in Mintos loans → interest earned (passive income).  
**Secondary:** Number of originators in portfolio → realized net return % (diversification).

**Methods:** From-scratch gradient descent (centered), scikit-learn, closed-form OLS, residual diagnostics, Monte-Carlo sensitivity.

**Platform context:** [Mintos](https://www.mintos.com/en/) is a MiFID II licensed European investment marketplace offering loans, bonds, ETFs, real estate and more. Top reasons people use it include competitive passive returns and easy diversification across originators and countries.

---

## Project Flowchart

![Flowchart](mintos_lr_flowchart.png)

### Audience adaptation
- Quantitative / technical readers → gradients, residual structure, estimator comparison.
- Retail / passive investors → effective yield in one sentence + R².
- Diversification-focused investors → pp lift per extra originator.
- Non-specialists → rising scatter line only.

**Disclaimer:** Synthetic teaching data inspired by publicly discussed return ranges. Not financial advice.


## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Load & Explore Capital → Interest

In [ ]:
df = pd.read_csv('data/mintos_portfolio.csv')
print('Shape:', df.shape)
display(df.head())
display(df.describe().round(2))

plt.figure(figsize=(7,5))
plt.scatter(df['capital_invested'], df['interest_earned'], alpha=0.55, edgecolor='k', linewidth=0.3, c='#0E6655')
plt.xlabel('Capital invested (€)')
plt.ylabel('Interest earned (€)')
plt.title('Mintos: Capital → Interest Earned (n=200)')
plt.tight_layout(); plt.show()

## 2. Points and Lines — Manual Prediction (warm-up)

In [ ]:
capital_sample = [500, 1200, 2500, 4000, 6000, 8000, 10000, 15000]
interest_obs   = [55, 130, 270, 420, 650, 880, 1100, 1650]

m, b = 0.11, 0
y_pred = [m * x + b for x in capital_sample]

plt.figure(figsize=(7,4))
plt.plot(capital_sample, interest_obs, 'o', label='Observed interest')
plt.plot(capital_sample, y_pred, '-', label=f'Manual line (m={m}, b={b})')
plt.xlabel('Capital (€)'); plt.ylabel('Interest (€)')
plt.title('Sample Portfolios — Manual Line (effective yield ≈ 11%)')
plt.legend(); plt.tight_layout(); plt.show()

print(f'Implied effective yield: {m*100:.1f}%')

## 3. Loss — Sum of Squared Errors

In [ ]:
x = [1000, 5000, 10000]
y = [110, 520, 1080]
m1, b1 = 0.10, 10
m2, b2 = 0.12, -20

y_pred1 = [m1*xi + b1 for xi in x]
y_pred2 = [m2*xi + b2 for xi in x]

total_loss1 = sum((yi - yh)**2 for yi, yh in zip(y, y_pred1))
total_loss2 = sum((yi - yh)**2 for yi, yh in zip(y, y_pred2))
print('Loss1:', total_loss1, 'Loss2:', total_loss2)
better_fit = 1 if total_loss1 < total_loss2 else 2
print('Better fit: line', better_fit)
print('Vectorised:', np.sum((np.array(y)-np.array(y_pred1))**2),
      np.sum((np.array(y)-np.array(y_pred2))**2))

## 4–5. Gradients for Intercept and Slope

In [ ]:
def get_gradient_at_b(x, y, m, b):
    N = len(x)
    diff = sum(y[i] - (m*x[i] + b) for i in range(N))
    return -2.0 / N * diff

def get_gradient_at_m(x, y, m, b):
    N = len(x)
    diff = sum(x[i] * (y[i] - (m*x[i] + b)) for i in range(N))
    return -2.0 / N * diff

def get_gradient_at_b_vec(x, y, m, b):
    return -2.0 * np.mean(y - (m*x + b))

def get_gradient_at_m_vec(x, y, m, b):
    return -2.0 * np.mean(x * (y - (m*x + b)))

print('Loop  gb:', get_gradient_at_b([1000,5000,10000],[110,520,1080],0.10,10))
print('Vec   gb:', get_gradient_at_b_vec(np.array([1000.,5000,10000]), np.array([110.,520,1080]), 0.10, 10))

## 6. One Gradient Step

In [ ]:
def step_gradient(x, y, b_current, m_current, learning_rate):
    b_grad = get_gradient_at_b(x, y, m_current, b_current)
    m_grad = get_gradient_at_m(x, y, m_current, b_current)
    b = b_current - learning_rate * b_grad
    m = m_current - learning_rate * m_grad
    return [b, m]

print('One step from (0,0):', step_gradient([1000,5000,10000], [110,520,1080], 0, 0, 1e-8))

## 7. Full Gradient Descent Loop

In [ ]:
def gradient_descent(x, y, learning_rate, num_iterations, return_history=False):
    b, m = 0.0, 0.0
    history = []
    x = list(x) if not isinstance(x, list) else x
    y = list(y) if not isinstance(y, list) else y
    for _ in range(num_iterations):
        b, m = step_gradient(x, y, b, m, learning_rate)
        if return_history:
            yhat = [m*xi + b for xi in x]
            loss = sum((yi-yh)**2 for yi,yh in zip(y,yhat)) / len(y)
            history.append((b, m, loss))
    if return_history:
        return [b, m], history
    return [b, m]

b_q, m_q = gradient_descent(capital_sample, interest_obs, 1e-8, 5000)
print(f'Sample series GD → m={m_q:.4f}, b={b_q:.2f}')

## 8. From-Scratch GD on Capital → Interest (centered)

In [ ]:
X = df['capital_invested'].values.astype(float)
y = df['interest_earned'].values.astype(float)

Xc = X - X.mean()
yc = y - y.mean()

# Centered data allows a more practical learning rate
(b_c, m_c), hist = gradient_descent(Xc.tolist(), yc.tolist(),
                                    learning_rate=1e-7,
                                    num_iterations=4000,
                                    return_history=True)
m_gd = m_c
b_gd = y.mean() - m_gd * X.mean()
print(f'Centered GD → m={m_gd:.4f}, b={b_gd:.2f}')
print(f'Effective yield: {m_gd*100:.2f}%  (interest per euro of capital)')

yhat_gd = m_gd * X + b_gd
mse_gd = np.mean((y - yhat_gd)**2)
ss_tot = np.sum((y - y.mean())**2)
r2_gd = 1 - np.sum((y - yhat_gd)**2) / ss_tot
print(f'MSE={mse_gd:.1f}, R²={r2_gd:.4f}')

plt.figure(figsize=(7,5))
plt.scatter(X, y, alpha=0.55, edgecolor='k', linewidth=0.3, c='#0E6655', label='Data')
xx = np.linspace(X.min()-100, X.max()+100, 100)
plt.plot(xx, m_gd*xx + b_gd, 'r-', lw=2.2, label=f'GD: interest={m_gd:.3f}·capital+{b_gd:.1f}')
plt.xlabel('Capital invested (€)'); plt.ylabel('Interest earned (€)')
plt.title('From-Scratch Gradient Descent Fit')
plt.legend(); plt.tight_layout(); plt.show()

losses = [h[2] for h in hist]
plt.figure(figsize=(6,3.5))
plt.plot(losses); plt.yscale('log')
plt.xlabel('Iteration'); plt.ylabel('MSE (centered)')
plt.title('Convergence of Centered GD'); plt.tight_layout(); plt.show()

## 9. scikit-learn LinearRegression

In [ ]:
X_2d = X.reshape(-1, 1)
model = LinearRegression().fit(X_2d, y)
m_sk, b_sk = model.coef_[0], model.intercept_
r2_sk = model.score(X_2d, y)
print(f'sklearn → m={m_sk:.4f}, b={b_sk:.2f}, R²={r2_sk:.4f}')
print(f'Effective yield: {m_sk*100:.2f}%')

yhat_sk = model.predict(X_2d)

plt.figure(figsize=(7,5))
plt.scatter(X, y, alpha=0.55, edgecolor='k', linewidth=0.3, c='#0E6655', label='Data')
plt.plot(xx, m_sk*xx + b_sk, 'g-', lw=2.2, label=f'sklearn: interest={m_sk:.3f}·capital+{b_sk:.1f}')
plt.xlabel('Capital invested (€)'); plt.ylabel('Interest earned (€)')
plt.title('scikit-learn LinearRegression Fit')
plt.legend(); plt.tight_layout(); plt.show()

## 10. More Practice — Closed Form, Residuals, Diversification

In [ ]:
# Closed-form OLS
x_bar, y_bar = X.mean(), y.mean()
m_cf = np.sum((X - x_bar)*(y - y_bar)) / np.sum((X - x_bar)**2)
b_cf = y_bar - m_cf * x_bar
print(f'Closed-form → m={m_cf:.4f}, b={b_cf:.2f}')

comparison = pd.DataFrame({
    'Method': ['Centered GD', 'sklearn', 'Closed-form'],
    'Slope m (effective yield)': [m_gd, m_sk, m_cf],
    'Intercept b': [b_gd, b_sk, b_cf]
})
display(comparison.round(4))

# Residual diagnostics
resid = y - yhat_sk
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(yhat_sk, resid, alpha=0.55, edgecolor='k', linewidth=0.3, c='#0E6655')
axes[0].axhline(0, color='r', ls='--')
axes[0].set_xlabel('Fitted interest (€)'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Fitted')
axes[1].hist(resid, bins=20, edgecolor='k', alpha=0.75, color='#0E6655')
axes[1].set_xlabel('Residual'); axes[1].set_title('Residual Distribution')
plt.suptitle(f'Residual Analysis (R² = {r2_sk:.3f})')
plt.tight_layout(); plt.show()
print('Residual mean ≈ 0?', round(resid.mean(), 6))

# --- Diversification: n_originators → realized_return_pct ---
div = pd.read_csv('data/mintos_diversification.csv')
N = div['n_originators'].values
R = div['realized_return_pct'].values
div_model = LinearRegression().fit(N.reshape(-1,1), R)
slope_div = div_model.coef_[0]
int_div = div_model.intercept_
r2_div = div_model.score(N.reshape(-1,1), R)
print(f'\nDiversification model: realized_return% = {int_div:.2f} + {slope_div:.4f}·n_originators')
print(f'Slope ≈ {slope_div:.3f} pp per extra originator  |  R² = {r2_div:.3f}')
print('Investor reading: each additional originator is associated with about {:.2f} pp higher realized net return.'.format(slope_div))

plt.figure(figsize=(7,4.5))
plt.scatter(N, R, alpha=0.6, edgecolor='k', linewidth=0.3, c='#1E8449')
xxn = np.linspace(N.min(), N.max(), 80)
plt.plot(xxn, int_div + slope_div*xxn, 'r-', lw=2, label=f'return% = {int_div:.1f} + {slope_div:.3f}·orig')
plt.xlabel('Number of originators in portfolio'); plt.ylabel('Realized net return (%)')
plt.title('Mintos Diversification: # Originators → Realized Return')
plt.legend(); plt.tight_layout(); plt.show()

## 11. Simulation — Learning Rate, Noise & Sample Size

In [ ]:
# === TUNABLE PARAMETERS ===
LEARNING_RATE = 1e-7
N_ITER        = 4000
NOISE_STD     = 0.0
SAMPLE_FRAC   = 1.0
N_REPS        = 30
# ===========================

def run_one(X, y, alpha, n_iter, noise_std, sample_frac):
    n = len(X)
    idx = np.random.choice(n, size=max(20, int(n*sample_frac)), replace=False)
    Xs, ys = X[idx], y[idx].copy()
    if noise_std > 0:
        ys = ys + np.random.normal(0, noise_std, size=len(ys))
    Xc, yc = Xs - Xs.mean(), ys - ys.mean()
    b_c, m_c = gradient_descent(Xc.tolist(), yc.tolist(), alpha, n_iter)
    m = m_c
    b = ys.mean() - m * Xs.mean()
    yhat = m*Xs + b
    r2 = 1 - np.sum((ys-yhat)**2) / np.sum((ys-ys.mean())**2)
    return m, b, r2

ms, bs, r2s = [], [], []
for _ in range(N_REPS):
    m, b, r2 = run_one(X, y, LEARNING_RATE, N_ITER, NOISE_STD, SAMPLE_FRAC)
    ms.append(m); bs.append(b); r2s.append(r2)

print(f'Settings: α={LEARNING_RATE}, iters={N_ITER}, noise={NOISE_STD}, frac={SAMPLE_FRAC}')
print(f'Slope (effective yield) mean±std : {np.mean(ms):.4f} ± {np.std(ms):.4f}')
print(f'Intercept               mean±std : {np.mean(bs):.2f} ± {np.std(bs):.2f}')
print(f'R²                      mean±std : {np.mean(r2s):.4f} ± {np.std(r2s):.4f}')

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].hist(ms, bins=12, edgecolor='k', alpha=0.75, color='#0E6655')
axes[0].axvline(m_sk, color='r', ls='--', label='sklearn'); axes[0].legend(fontsize=8)
axes[0].set_title('Slope (effective yield)')
axes[1].hist(bs, bins=12, edgecolor='k', alpha=0.75, color='#0E6655')
axes[1].axvline(b_sk, color='r', ls='--'); axes[1].set_title('Intercept')
axes[2].hist(r2s, bins=12, edgecolor='k', alpha=0.75, color='#0E6655')
axes[2].axvline(r2_sk, color='r', ls='--'); axes[2].set_title('R²')
plt.suptitle('Monte-Carlo Sensitivity — Capital → Interest')
plt.tight_layout(); plt.show()

## Cheat Sheet

| Concept | Formula / Code | Mintos reading |
|---------|----------------|----------------|
| Line | `yhat = m*x + b` | Predicted interest or realized return % |
| Slope `m` | marginal effect | Effective yield; or pp return per extra originator |
| Intercept `b` | baseline | Use cautiously outside observed range |
| SSE / MSE | `sum((y-yhat)**2)` / mean | Total / average squared prediction error |
| Gradient b / m | `−2/N · Σ …` | Direction of steepest ascent of loss |
| Update | `param -= α * gradient` | Step downhill on the loss surface |
| Convergence | params almost stop changing | Approximate minimum of loss reached |
| sklearn | `LinearRegression().fit` | Fast reference OLS |
| Closed form | `m = cov(x,y)/var(x)` | Exact simple-OLS solution |
| Centering | GD on demeaned data | Essential when capital is large |

**Key results (capital → interest)**
- Effective yield ≈ **10.9%** (interest per euro of capital)
- Intercept ≈ **−13**
- R² ≈ **0.59**
- GD, sklearn and closed-form agree closely when learning rate and iterations are sensible.

**Diversification practice**
- ≈ **0.13 percentage points** higher realized net return per extra originator in the portfolio.

**Disclaimer:** Synthetic teaching data. Not financial advice. Investing involves risk of loss.
